This is where we convert all 4,718 scheme texts into vectors and build the FAISS search index.

In [10]:
import json
import numpy as np
import faiss
import os
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

# Paths
SCHEMES_PATH    = "../data/schemes.json"
EMBEDDINGS_PATH = "../embeddings/scheme_embeddings.npy"
INDEX_PATH      = "../embeddings/faiss_index.bin"
METADATA_PATH   = "../embeddings/metadata.json"

os.makedirs("../embeddings", exist_ok=True)

print("✅ Imports done")

✅ Imports done


LOading the schemes

In [4]:
with open(SCHEMES_PATH, encoding="utf-8") as f:
    schemes = json.load(f)

texts = [s["full_text"] for s in schemes]

print(f"✅ Loaded {len(schemes)} schemes")
print(f"\n🔍 Sample full_text:\n{texts[0][:300]}")

✅ Loaded 4718 schemes

🔍 Sample full_text:
Scheme Name: Stand-Up India
Ministry: Ministry Of Finance
State: All
Level: Central
Category: Business & Entrepreneurship, Banking,Financial Services and Insurance, Social welfare & Empowerment
Description: A scheme by Ministry of Finance for financing SC/ST and  Women Entrepreneurs by facilitating 


Load MuRIL Model

In [5]:
print("⏳ Loading MuRIL model (downloads ~500MB on first run, be patient)...")

model = SentenceTransformer("google/muril-base-cased")

print("✅ MuRIL model loaded!")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

⏳ Loading MuRIL model (downloads ~500MB on first run, be patient)...


c:\Users\knlwa\Desktop\GIT\SAHAYAK_AI\sahayak\sahayak_env\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\knlwa\.cache\huggingface\hub\models--google--muril-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 55341.90it/s]
[transformers] Be

✅ MuRIL model loaded!
   Embedding dimension: 768


C:\Users\knlwa\AppData\Local\Temp\ipykernel_2696\1641080377.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")


Generate Embeddings

In [6]:
print(f"⏳ Embedding {len(texts)} schemes...")
print("   This will take 5-15 minutes depending on your CPU. Grab a chai ☕")

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings generated!")
print(f"   Shape: {embeddings.shape}")
# Expected: (4718, 768)

np.save(EMBEDDINGS_PATH, embeddings)
print(f"✅ Embeddings saved to {EMBEDDINGS_PATH}")

⏳ Embedding 4718 schemes...
   This will take 5-15 minutes depending on your CPU. Grab a chai ☕


Batches: 100%|██████████| 148/148 [07:04<00:00,  2.86s/it]


✅ Embeddings generated!
   Shape: (4718, 768)
✅ Embeddings saved to ../embeddings/scheme_embeddings.npy


Build FAISS Index

In [11]:
print("⏳ Building FAISS index...")

embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
dimension  = embeddings.shape[1]  # 768

# Normalize vectors for cosine similarity
faiss.normalize_L2(embeddings)

# Build flat index (exact search — best for under 100k vectors)
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product (cosine after normalization)
index.add(embeddings)

print(f"✅ FAISS index built!")
print(f"   Total vectors: {index.ntotal}")
print(f"   Dimension: {dimension}")

faiss.write_index(index, INDEX_PATH)
print(f"✅ Index saved to {INDEX_PATH}")

⏳ Building FAISS index...
✅ FAISS index built!
   Total vectors: 4718
   Dimension: 768
✅ Index saved to ../embeddings/faiss_index.bin


Save Metadata

In [13]:
metadata = [
    {
        "id":       s["id"],
        "title":    s["title"],
        "ministry": s["ministry"],
        "state":    s["state"],
        "url":      s["url"],
        "category": s.get("category", []),
        "tags":     s.get("tags", [])
    }
    for s in schemes
]

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata saved — {len(metadata)} entries")

✅ Metadata saved — 4718 entries


Test Search

In [15]:
def search(query, top_k=5):
    query_vec = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)
    
    scores, indices = index.search(query_vec, k=top_k)
    
    print(f"\n Query: '{query}'")
    print("-" * 50)
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        m = metadata[idx]
        print(f"{rank+1}. {m['title']}")
        print(f"   State: {m['state']} | Score: {score:.3f}")
        print(f"   URL: {m['url']}")
        print()

# Test with different queries
search("scheme for farmers in Maharashtra")
search("scholarship for students")
search("pension for elderly women")
search("startup funding for entrepreneurs")


 Query: 'scheme for farmers in Maharashtra'
--------------------------------------------------
1. Nanaji Deshmukh Krishi Sanjivani Prakalp
   State: Maharashtra | Score: 0.995
   URL: https://www.myscheme.gov.in/schemes/pocra

2. State Millet Mission
   State: Uttarakhand | Score: 0.995
   URL: https://www.myscheme.gov.in/schemes/smm

3. Chief Minister’s White Revolutions Scheme
   State: Arunachal Pradesh | Score: 0.995
   URL: https://www.myscheme.gov.in/schemes/cmwrs

4. Distribution of Seeds of more productions varieties/ Hybrids varieties Seeds and fertilizer at subsidies etc. to Adivasi farmers in Tribal Area: Plant protection pesticides Gujarat
   State: Gujarat | Score: 0.995
   URL: https://www.myscheme.gov.in/schemes/dsmpvsfpppguj

5. Sardar Krushi Jyoti Yojana
   State: Gujarat | Score: 0.995
   URL: https://www.myscheme.gov.in/schemes/skjy


 Query: 'scholarship for students'
--------------------------------------------------
1. Prabhuddha Overseas Scholarship
   State: Ka